Import required libraries

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

Dataset for english and french pairs

In [3]:
pairs = [
    ("i love cats", "j aime les chats"),
    ("i love dogs", "j aime les chiens"),
    ("i like cats", "j aime les chats"),
    ("i like dogs", "j aime les chiens"),
    ("i see cats", "je vois les chats"),
    ("i see dogs", "je vois les chiens"),
    ("you love cats", "tu aimes les chats"),
    ("you love dogs", "tu aimes les chiens"),
    ("you like cats", "tu aimes les chats"),
    ("you like dogs", "tu aimes les chiens"),
]

create a vocabulary function for both input and target seqence

In [4]:
def build_vocab(sentences):

    vocab = {
        "<PAD>": 0,
        "<SOS>": 1,
        "<EOS>": 2,
        "<UNK>": 3
    }

    for sentence in sentences:
        for word in sentence.split():
            if word not in vocab:
                vocab[word] = len(vocab)

    return vocab


src_vocab = build_vocab([src for src, tgt in pairs])
tgt_vocab = build_vocab([tgt for src, tgt in pairs])

print("English vocab:", src_vocab)
print("French vocab:", tgt_vocab)


English vocab: {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3, 'i': 4, 'love': 5, 'cats': 6, 'dogs': 7, 'like': 8, 'see': 9, 'you': 10}
French vocab: {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3, 'j': 4, 'aime': 5, 'les': 6, 'chats': 7, 'chiens': 8, 'je': 9, 'vois': 10, 'tu': 11, 'aimes': 12}


tokenize input and target sequence

In [5]:
def encode_sentence(sentence, vocab):

    tokens = ["<SOS>"] + sentence.split() + ["<EOS>"]

    return [
        vocab.get(token, vocab["<UNK>"])
        for token in tokens
    ]


src_sequences = [
    encode_sentence(src, src_vocab)
    for src, tgt in pairs
]

tgt_sequences = [
    encode_sentence(tgt, tgt_vocab)
    for src, tgt in pairs
]

convert the seq to tensors

In [6]:
src_tensor = torch.tensor(src_sequences)
tgt_tensor = torch.tensor(tgt_sequences)

print("Source shape:", src_tensor.shape)
print("Target shape:", tgt_tensor.shape)

Source shape: torch.Size([10, 5])
Target shape: torch.Size([10, 6])


Create an encoding class to get the hidden representation of the input sequence as well as the context vector.

In [7]:
class Encoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True
        )

    def forward(self, x):

        # x:
        # [batch, source_length]

        embedded = self.embedding(x)

        # embedded:
        # [batch, source_length, embedding_dim]

        outputs, (hidden, cell) = self.lstm(embedded)

        # outputs:
        # [batch, source_length, hidden_dim]

        # hidden:
        # [1, batch, hidden_dim]

        # cell:
        # [1, batch, hidden_dim]

        return outputs, hidden, cell

below is the bahdanau attention implementation to get attention weights

In [8]:
class BahdanauAttention(nn.Module):

    def __init__(
        self,
        encoder_hidden_dim,
        decoder_hidden_dim,
        attention_dim
    ):
        super().__init__()

        # U_a h_i
        self.encoder_projection = nn.Linear(
            encoder_hidden_dim,
            attention_dim
        )

        # W_a s_(t-1)
        self.decoder_projection = nn.Linear(
            decoder_hidden_dim,
            attention_dim
        )

        # v_a^T
        self.energy = nn.Linear(
            attention_dim,
            1,
            bias=False
        )

    def forward(
        self,
        decoder_hidden,
        encoder_outputs
    ):

        # decoder_hidden:
        # [1, batch, decoder_hidden_dim]

        # Remove the first dimension
        decoder_hidden = decoder_hidden.squeeze(0)

        # Now:
        # [batch, decoder_hidden_dim]


        # ----------------------------------------------------
        # Project encoder hidden states
        # ----------------------------------------------------

        encoder_features = self.encoder_projection(
            encoder_outputs
        )

        # [batch, source_length, attention_dim]


        # ----------------------------------------------------
        # Project decoder hidden state
        # ----------------------------------------------------

        decoder_features = self.decoder_projection(
            decoder_hidden
        )

        # [batch, attention_dim]


        # Add sequence dimension
        decoder_features = decoder_features.unsqueeze(1)

        # [batch, 1, attention_dim]


        # ----------------------------------------------------
        # Bahdanau energy
        # ----------------------------------------------------

        energy = torch.tanh(
            encoder_features + decoder_features
        )

        # [batch, source_length, attention_dim]


        # ----------------------------------------------------
        # v_a^T
        # ----------------------------------------------------

        energy = self.energy(energy)

        # [batch, source_length, 1]

        energy = energy.squeeze(-1)

        # [batch, source_length]


        # ----------------------------------------------------
        # Convert scores into attention weights
        # ----------------------------------------------------

        attention_weights = torch.softmax(
            energy,
            dim=1
        )

        # [batch, source_length]

        return attention_weights

Implementing the decoder for bahdanau attention

In [9]:
class Decoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        encoder_hidden_dim,
        decoder_hidden_dim,
        attention_dim
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        self.attention = BahdanauAttention(
            encoder_hidden_dim,
            decoder_hidden_dim,
            attention_dim
        )

        self.lstm = nn.LSTM(
            embedding_dim + encoder_hidden_dim,
            decoder_hidden_dim,
            batch_first=True
        )

        self.output_layer = nn.Linear(
            decoder_hidden_dim + encoder_hidden_dim,
            vocab_size
        )

    def forward(
        self,
        input_token,
        hidden,
        cell,
        encoder_outputs
    ):

        # input_token:
        # [batch, 1]

        embedded = self.embedding(input_token)

        # [batch, 1, embedding_dim]


        # ====================================================
        # BAHDAUNAU ATTENTION
        # ====================================================

        attention_weights = self.attention(
            hidden,
            encoder_outputs
        )

        # [batch, source_length]


        # Add dimension for matrix multiplication

        attention_weights = attention_weights.unsqueeze(1)

        # [batch, 1, source_length]


        # ====================================================
        # CONTEXT VECTOR
        # ====================================================

        context = torch.bmm(
            attention_weights,
            encoder_outputs
        )

        # [batch, 1, encoder_hidden_dim]


        # ====================================================
        # LSTM INPUT
        # ====================================================

        lstm_input = torch.cat(
            [embedded, context],
            dim=2
        )

        # [batch, 1,
        #  embedding_dim + encoder_hidden_dim]


        output, (hidden, cell) = self.lstm(
            lstm_input,
            (hidden, cell)
        )

        # ====================================================
        # PREDICTION
        # ====================================================

        prediction = self.output_layer(
            torch.cat(
                [output, context],
                dim=2
            )
        )

        # prediction:
        # [batch, 1, target_vocab_size]

        return (
            prediction,
            hidden,
            cell,
            attention_weights
        )

Below is the implementation of the seq2seq layer that combines the encoder-decoder layer for i/o task

In [10]:
class Seq2Seq(nn.Module):

    def __init__(self, encoder, decoder):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder

    def forward(
        self,
        source,
        target,
        teacher_forcing_ratio=0.5
    ):

        batch_size = source.shape[0]
        target_length = target.shape[1]
        target_vocab_size = self.decoder.output_layer.out_features

        outputs = torch.zeros(
            batch_size,
            target_length,
            target_vocab_size
        )

        # ----------------------------------------------------
        # ENCODER
        # ----------------------------------------------------

        encoder_outputs, hidden, cell = self.encoder(source)

        # ----------------------------------------------------
        # Start decoder with <SOS>
        # ----------------------------------------------------

        decoder_input = target[:, 0].unsqueeze(1)

        # ----------------------------------------------------
        # Generate target sequence
        # ----------------------------------------------------

        for t in range(1, target_length):

            prediction, hidden, cell, attention = self.decoder(
                decoder_input,
                hidden,
                cell,
                encoder_outputs
            )

            outputs[:, t:t+1] = prediction

            # Teacher forcing

            teacher_force = (
                random.random() < teacher_forcing_ratio
            )

            predicted_token = prediction.argmax(
                dim=2
            )

            if teacher_force:

                decoder_input = target[:, t].unsqueeze(1)

            else:

                decoder_input = predicted_token

        return outputs

Create a model for training, by setting the hyperparameters.

In [12]:
INPUT_DIM = len(src_vocab)
OUTPUT_DIM = len(tgt_vocab)

EMBEDDING_DIM = 64
ENCODER_HIDDEN_DIM = 128
DECODER_HIDDEN_DIM = 128
ATTENTION_DIM = 64


encoder = Encoder(
    INPUT_DIM,
    EMBEDDING_DIM,
    ENCODER_HIDDEN_DIM
)

decoder = Decoder(
    OUTPUT_DIM,
    EMBEDDING_DIM,
    ENCODER_HIDDEN_DIM,
    DECODER_HIDDEN_DIM,
    ATTENTION_DIM
)

model = Seq2Seq(
    encoder,
    decoder
)

Train the model

In [13]:
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

criterion = nn.CrossEntropyLoss(
    ignore_index=tgt_vocab["<PAD>"]
)


EPOCHS = 1000

for epoch in range(EPOCHS):

    model.train()

    optimizer.zero_grad()

    output = model(
        src_tensor,
        tgt_tensor,
        teacher_forcing_ratio=0.5
    )

    # Remove <SOS>
    output = output[:, 1:, :]

    target = tgt_tensor[:, 1:]

    # [batch, sequence, vocab]
    # ->
    # [batch * sequence, vocab]

    output = output.reshape(
        -1,
        OUTPUT_DIM
    )

    target = target.reshape(-1)

    loss = criterion(
        output,
        target
    )

    loss.backward()

    optimizer.step()

    if (epoch + 1) % 100 == 0:

        print(
            f"Epoch {epoch+1}/{EPOCHS} "
            f"Loss: {loss.item():.4f}"
        )

Epoch 100/1000 Loss: 0.0128
Epoch 200/1000 Loss: 0.0040
Epoch 300/1000 Loss: 0.0021
Epoch 400/1000 Loss: 0.0013
Epoch 500/1000 Loss: 0.0009
Epoch 600/1000 Loss: 0.0007
Epoch 700/1000 Loss: 0.0005
Epoch 800/1000 Loss: 0.0004
Epoch 900/1000 Loss: 0.0003
Epoch 1000/1000 Loss: 0.0003


Below is a function for the translation task

In [14]:
def translate_with_attention(
    model,
    sentence,
    src_vocab,
    tgt_vocab,
    max_length=10
):

    model.eval()

    source_tokens = encode_sentence(
        sentence,
        src_vocab
    )

    source = torch.tensor(
        [source_tokens]
    )

    with torch.no_grad():

        encoder_outputs, hidden, cell = model.encoder(
            source
        )

        decoder_input = torch.tensor(
            [[tgt_vocab["<SOS>"]]]
        )

        generated_tokens = []

        attention_history = []

        for _ in range(max_length):

            prediction, hidden, cell, attention = model.decoder(
                decoder_input,
                hidden,
                cell,
                encoder_outputs
            )

            predicted_token = prediction.argmax(
                dim=2
            ).item()

            generated_tokens.append(
                predicted_token
            )

            attention_history.append(
                attention.squeeze(0).squeeze(0).numpy()
            )

            if predicted_token == tgt_vocab["<EOS>"]:
                break

            decoder_input = torch.tensor(
                [[predicted_token]]
            )

    # Convert IDs back to words

    reverse_vocab = {
        index: token
        for token, index in tgt_vocab.items()
    }

    translated_words = [
        reverse_vocab[token]
        for token in generated_tokens
    ]

    return translated_words, attention_history

A translation task, printing the translation sequence and the attention weights

In [16]:
translation, attention = translate_with_attention(
    model,
    "i love cats",
    src_vocab,
    tgt_vocab
)

print("Translation:")
print(translation)

print("\nAttention:")
for step, weights in enumerate(attention):
    print(step, weights)

Translation:
['j', 'aime', 'les', 'chats', '<EOS>']

Attention:
0 [1.5331265e-04 2.3549057e-03 2.9181831e-02 1.1195645e-01 8.5635352e-01]
1 [1.3994818e-04 7.9243595e-04 9.2680240e-03 3.8623303e-02 9.5117629e-01]
2 [0.10002515 0.20095718 0.22738752 0.23114078 0.24048938]
3 [4.2575670e-05 2.1406708e-04 4.8446502e-03 2.9836794e-02 9.6506196e-01]
4 [0.10220148 0.20352453 0.22910893 0.23004854 0.23511653]
